# Interaction and Selection

The WebGPU scene provides built-in camera controls (rotate, pan, zoom) automatically.
This notebook shows how to add **selection** — running callbacks when the user clicks
on a rendered object.

> **Note:** Selection requires a live Python kernel and only works in interactive
> Jupyter sessions. Exported HTML pages (via `nbconvert` or Sphinx) retain camera
> interaction but do not support selection callbacks.

## Basic selection

To enable selection:

1. Call `scene.select(x, y)` when the user clicks on the canvas.
2. Register callbacks via `renderer.on_select(callback)` — the scene finds which
   renderer is at the clicked pixel and triggers its callback.

The callback receives a `SelectEvent` with `uint32` and `float32` views of the
selection buffer (4 × u32 per pixel).

In [ ]:
import webgpu.jupyter as wj
import numpy as np
from webgpu.shapes import ShapeRenderer, generate_cylinder, generate_cone
from webgpu import Colormap, Labels, Colorbar

rand = np.random.random
N = 10
thickness = 0.1
length = 0.3

cylinder = generate_cylinder(8, thickness)
cone = generate_cone(8, thickness)

cmap_cone = Colormap(10, 11, "viridis")
cmap_cyl = Colormap(0, 1)

cylinders = ShapeRenderer(
    cylinder, rand((N, 3)), length * rand((N, 3)), rand(2 * N), colormap=cmap_cyl
)
cones = ShapeRenderer(
    cone, rand((N, 3)), length * rand((N, 3)), 10.0 + rand(2 * N), colormap=cmap_cone
)

label = Labels(["Click an object"], [[-1, -1]], font_size=16)
scene = wj.Draw([cylinders, cones, label, Colorbar(cmap_cone), Colorbar(cmap_cyl, (-0.9, 0.75))])


def set_label(s):
    label.labels[0] = s
    label.set_needs_update()
    scene.render()


def on_select_cone(ev):
    set_label(f"Selected cone {ev.uint32[0]} with value {ev.float32[1]}")


def on_select_cyl(ev):
    set_label(f"Selected cylinder {ev.uint32[0]} with value {ev.float32[1]}")


cones.on_select(on_select_cone)
cylinders.on_select(on_select_cyl)


def on_click(ev):
    if ev["button"] == 0:
        scene.select(ev["canvasX"], ev["canvasY"])


scene.input_handler.on_click(on_click)

## How selection works

Each renderer that supports selection has **two** rendering pipelines:

- The **rendering pipeline** draws the object normally.
- The **selection pipeline** renders object metadata encoded as pixel colors into
  a `vec4<u32>` selection buffer.

The selection pipeline uses the same vertex shader but a **different fragment shader**.
The default `ShapeRenderer` selection shader writes:

| Channel | Content |
|---------|---------|
| `x` | `@RENDER_OBJECT_ID@` — identifies which renderer |
| `y` | `input.instance` — which instance (e.g. which cylinder) |
| `z` | `bitcast<u32>(input.value.x)` — scalar value as bits |
| `w` | `0` |

WGSL excerpt:

```wgsl
@fragment fn shape_fragment_main_select(
    input: ShapeVertexOut,
) -> @location(0) vec4<u32> {
    return vec4<u32>(@RENDER_OBJECT_ID@, input.instance,
                     bitcast<u32>(input.value.x), 0);
}
```

When `scene.select(x, y)` is called, the scene reads the selection buffer at that
pixel, identifies the renderer from the object ID, and calls its `on_select` callback
with the raw `vec4<u32>` data.

## Custom selection shader

You can override `select_entry_point` and `get_shader_code()` to store different
data in the selection buffer. The example below stores the 3D position of the
clicked point instead of instance index + value.

In [ ]:
select_shader = """
@fragment fn my_select_shader(
    input: ShapeVertexOut,
) -> @location(0) vec4<u32> {
    return vec4<u32>(@RENDER_OBJECT_ID@, bitcast<vec3<u32>>(input.p));
}
"""


class MyShapeRenderer(ShapeRenderer):
    select_entry_point = "my_select_shader"

    def get_shader_code(self):
        return super().get_shader_code() + select_shader


mycylinders = MyShapeRenderer(cylinder, rand((N, 3)), rand((N, 3)), rand(2 * N))
mylabel = Labels([""], [[-1, -0.8]], font_size=14)


def on_select_mycyl(ev):
    mylabel.labels[0] = f"Selected at point {ev.float32}"
    mylabel.set_needs_update()
    myscene.render()


mycylinders.on_select(on_select_mycyl)

myscene = wj.Draw([mycylinders, mylabel])
myscene.input_handler.on_click(lambda ev: myscene.select(ev["canvasX"], ev["canvasY"]))